# 03 · Masked event transformer + XGBoost
Run sections 1–3 to verify the encoder and the five saved tree models. Sections 4–5 train the encoder and rebuild the recommendation model. The delivered variant uses 3:1 column sampling.

## 1. Data and checkpoints

In [ ]:
from pathlib import Path
import json
import math

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'environment/shared.ipynb').exists())
def load_cells(path):
    for cell in json.loads(path.read_text())['cells']:
        if cell['cell_type'] == 'code':
            exec(''.join(cell['source']), globals())

load_cells(ROOT / 'environment/shared.ipynb')
import torch
train = pd.read_csv(ROOT / 'data/raw/train.csv', dtype={'event': str, 'user': str})
test = pd.read_csv(ROOT / 'data/raw/test.csv', dtype={'event': str, 'user': str})
samples = pd.concat([train.drop(columns=['interested', 'not_interested']), test], ignore_index=True)
folds = pd.read_csv(ROOT / 'data/folds.csv').fold.to_numpy()
base = pd.read_pickle(ROOT / '02_xgboost_baseline/reference/features.pkl')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(4)
HERE = ROOT / '03_masked_event_transformer'
reference = HERE / 'reference'
output = HERE / 'results'
output.mkdir(exist_ok=True)
load_cells(ROOT / '03_masked_event_transformer/event_encoder.ipynb')
event_ref = ROOT / '03_masked_event_transformer/reference'
meta = json.loads((event_ref / 'encoder.json').read_text())
teacher = build_model(meta['cards']).to(device)
teacher.load_state_dict(torch.load(event_ref / 'encoder.pt', map_location=device))
teacher.eval()
event_ids = pd.Index(pd.read_csv(event_ref / 'candidate_residual_events.csv', dtype=str).event)
row_index = event_ids.get_indexer(samples.event)

## 2. Check the representation on every candidate event

In [ ]:
def add_residuals(base, residuals):
    columns = [f'pretrained_t{token:02d}_r{dim:03d}' for token in range(10) for dim in range(128)]
    return pd.concat([base, pd.DataFrame(residuals[row_index], columns=columns, index=base.index)], axis=1)
saved = np.load(reference / 'candidate_residuals.npy')
restored = np.concatenate(list(residual_batches(teacher, np.load(reference / 'candidate_codes.npy'))))
print('Maximum residual difference', float(np.abs(restored - saved).max()))
x = add_residuals(base, saved)

## 3. Score the five saved folds
Evaluate the original MPS inputs and then the coordinates regenerated on this device. Small floating-point differences can move values across a tree split.

In [ ]:
scores = evaluate_saved(x, train, folds, reference, [42], baseline=True)
expected = pd.read_csv(reference / 'oof.csv').probability
print('OOF MAP@200', map_users(train, scores).ap200.mean())
print('Maximum probability difference', float(np.abs(scores - expected).max()))
regenerated = add_residuals(base, restored)
end_to_end = evaluate_saved(regenerated, train, folds, reference, [42], baseline=True)
print('MAP@200 with regenerated neural inputs', map_users(train, end_to_end).ap200.mean())


## 4. Retrain the event model for one pass over the catalog
Use the packaged group definition and fitted bin edges. Each event contributes three masked token targets.

In [ ]:
def pretrain(codes, cards):
    torch.manual_seed(42)
    model = build_model(cards).to(device)
    order = np.random.default_rng(42).permutation(len(codes))
    batch = 1024
    total_steps = math.ceil(len(codes) / batch)
    warmup = math.ceil(0.05 * total_steps)
    optimizer = torch.optim.AdamW([{'params': [p for p in model.parameters() if p.ndim > 1], 'weight_decay': 0.01}, {'params': [p for p in model.parameters() if p.ndim == 1], 'weight_decay': 0.0}], lr=3e-4)
    rng = torch.Generator().manual_seed(42)
    for step in range(total_steps):
        fields = torch.from_numpy(np.asarray(codes[order[step * batch:(step + 1) * batch]], dtype=np.int64)).to(device)
        selected = torch.zeros((len(fields), 10), dtype=torch.bool).scatter_(1, torch.rand((len(fields), 10), generator=rng).argsort(dim=1)[:, :3], True).to(device)
        lr = 3e-4 * (step + 1) / warmup if step < warmup else 3e-5 + 0.5 * (3e-4 - 3e-5) * (1 + math.cos(math.pi * (step - warmup + 1) / (total_steps - warmup)))
        for group in optimizer.param_groups:
            group['lr'] = lr
        optimizer.zero_grad(set_to_none=True)
        _, loss, _ = model(fields, selected)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return model.eval()

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory(dir=output) as scratch:
    codes, catalog_ids = encode_catalog(ROOT / 'data/raw/events.csv.gz', meta, Path(scratch))
    trained = pretrain(codes, meta['cards'])
    torch.save(trained.state_dict(), output / 'encoder.pt')
    candidate_codes = codes[catalog_ids.get_indexer(event_ids.astype('uint64'))]
    learned = np.concatenate(list(residual_batches(trained, candidate_codes)))

## 5. Rebuild the candidate table and retrain five folds
Each fold uses its recorded feature sampling weights after category encoding.

In [ ]:
original, _ = features(ROOT / 'data/raw', samples)
x = add_residuals(original, learned)
parameters = json.loads((HERE / 'parameters.json').read_text())
print(train_folds(x, train, test, folds, parameters, [42], output, weight_directory=reference))